# EDA — Heat Pump Grid Load Prediction (Task 3)

**Cel:** Przewidzieć miesięczną średnią `x2` (wskaźnik obciążenia sieci) dla 600 pomp ciepła, maj–październik 2025.  
**Dane treningowe:** październik 2024 – kwiecień 2025 (zima).  
**Wyzwanie:** Out-of-Distribution — trenujemy na zimie, predykcja na lecie. Pompy ciepła latem pracują tylko w trybie CWU (ciepła woda użytkowa), więc obciążenie spada do ~10–20% wartości zimowych.

In [1]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

DATA_DIR = Path("data")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (12, 4)

## 1. devices.csv — metadane urządzeń

In [ ]:
devices = pl.read_csv(DATA_DIR / "devices.csv")
print(f"Urządzenia: {len(devices)}")
devices.head()

In [ ]:
# Rozkład geograficzny — Polska (bbox: lat 49-56, lon 14-25)
lat = devices["latitude"].to_numpy()
lon = devices["longitude"].to_numpy()

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(lon, lat, alpha=0.4, s=20, c="steelblue", edgecolors="none")
ax.set_xlim(13.5, 25.0)
ax.set_ylim(49.0, 55.5)
ax.set_xlabel("Długość geograficzna")
ax.set_ylabel("Szerokość geograficzna")
ax.set_title("Lokalizacje 600 pomp ciepła (Polska)")
ax.grid(True, alpha=0.3)
# Informacja o zakresie
print(f"Zakres lat: {lat.min():.1f} – {lat.max():.1f}")
print(f"Zakres lon: {lon.min():.1f} – {lon.max():.1f}")
print(
    f"Wszystkie urządzenia w granicach Polski: {((lat > 49) & (lat < 55.5) & (lon > 14) & (lon < 24.5)).all()}"
)
plt.tight_layout()
plt.show()

# Ile unikalnych lokalizacji po zaokrągleniu do 0.1°
unique_locs = (
    devices.with_columns(
        [
            pl.col("latitude").round(1),
            pl.col("longitude").round(1),
        ]
    )
    .select(["latitude", "longitude"])
    .unique()
)
print(f"Unikalne lokalizacje pogodowe (zaokrąglone 0.1°): {len(unique_locs)}")

## 2. data.csv — telemetria 5-minutowa

Plik ma ~10.4 GB i 64.5M wierszy. Wczytujemy sample dla EDA.

In [ ]:
# Wczytujemy lazy i pobieramy sample: pierwsze 2 urządzenia, cały zakres
sample_devices = devices["deviceId"][:2].to_list()

lazy = pl.scan_csv(DATA_DIR / "data.csv", try_parse_dates=False)
sample = (
    lazy.filter(pl.col("deviceId").is_in(sample_devices))
    .with_columns(
        pl.col("timedate")
        .str.replace(r" UTC$", "")
        .str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
    )
    .sort(["deviceId", "timedate"])
    .collect()
)

print(f"Sample: {len(sample):,} wierszy, {sample['deviceId'].n_unique()} urządzenia")
print(f"Kolumny: {sample.columns}")
sample.head(3)

In [ ]:
# Sprawdzamy okresy (periods)
print("Rozkład period:")
print(sample.group_by("period").agg(pl.len().alias("n")).sort("period"))

In [ ]:
# Statystyki x2 w train
train = sample.filter(pl.col("period") == "train")
print("Statystyki x2 (train):")
print(train.select("x2").describe())
print(f"\nProcent x2 == 0: {(train['x2'] == 0).mean():.1%}")
print(
    f"x2 jest NULL w validation/test: {sample.filter(pl.col('period') != 'train')['x2'].is_null().mean():.1%}"
)

## 3. Rozkład x2 — cel predykcji

In [ ]:
x2_vals = train["x2"].drop_nulls().to_numpy()
x2_nonzero = x2_vals[x2_vals > 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(x2_vals, bins=80, color="steelblue", edgecolor="none", alpha=0.8)
axes[0].set_title("Rozkład x2 (wszystkie, train)")
axes[0].set_xlabel("x2")
axes[0].set_ylabel("Liczba obserwacji")

axes[1].hist(np.log1p(x2_nonzero), bins=80, color="tomato", edgecolor="none", alpha=0.8)
axes[1].set_title("Rozkład log1p(x2) — x2 > 0")
axes[1].set_xlabel("log1p(x2)")

plt.tight_layout()
plt.show()
print("x2 jest prawostronnie asymetryczny → log-transformacja poprawia modelowanie")

## 4. Przebieg czasowy x2 — widoczna sezonowość

In [ ]:
dev0 = sample.filter(pl.col("deviceId") == sample_devices[0]).sort("timedate")

# Dzienne mediany
daily = (
    dev0.with_columns(pl.col("timedate").dt.date().alias("date"))
    .group_by("date")
    .agg(
        [
            pl.col("x2").median().alias("x2_med"),
            pl.col("t1").mean().alias("t1_mean"),
            pl.col("period").first(),
        ]
    )
    .sort("date")
)

dates = daily["date"].to_numpy()
x2_med = daily["x2_med"].to_numpy()
t1_mean = daily["t1_mean"].to_numpy()
periods = daily["period"].to_numpy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# x2
mask_train = periods == "train"
mask_val = periods == "validation"
mask_test = periods == "test"

ax1.plot(dates[mask_train], x2_med[mask_train], color="steelblue", lw=1, label="train (x2 znane)")
if mask_val.any():
    ax1.axvline(
        x=dates[mask_val].min(),
        color="orange",
        ls="--",
        lw=1.5,
        label="→ validation (maj-cze 2025)",
    )
if mask_test.any():
    ax1.axvline(
        x=dates[mask_test].min(), color="red", ls="--", lw=1.5, label="→ test (lip-paź 2025)"
    )
ax1.set_ylabel("x2 (dzienna mediana)")
ax1.set_title(f"Urządzenie: {sample_devices[0][:16]}… — przebieg x2")
ax1.legend()

# t1 (znormalizowana temp zewnętrzna)
ax2.plot(dates, t1_mean, color="tomato", lw=1)
ax2.set_ylabel("t1 (znorm. temp. zewnętrzna)")
ax2.set_xlabel("Data")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.tight_layout()
plt.show()

## 5. Korelacja t1 (temp. zewnętrzna) z x2

To jest serce problemu: zima → wyższe x2 gdy niższe t1. Latem t1 rośnie i x2 spada — ale model musi to *ekstrapolować*, bo tego na treningu nie widział.

In [ ]:
# Sample z kilku urządzeń dla lepszego obrazu
more_devices = devices["deviceId"][:10].to_list()
multi = (
    lazy.filter(pl.col("deviceId").is_in(more_devices) & (pl.col("period") == "train"))
    .select(["deviceId", "t1", "x2"])
    .collect()
    .filter(pl.col("x2").is_not_null() & pl.col("t1").is_not_null())
)

fig, ax = plt.subplots(figsize=(10, 5))
for did in more_devices:
    sub = multi.filter(pl.col("deviceId") == did)
    ax.scatter(sub["t1"].to_numpy(), sub["x2"].to_numpy(), alpha=0.05, s=2)

ax.set_xlabel("t1 (znorm. temp. zewnętrzna)")
ax.set_ylabel("x2 (obciążenie sieci)")
ax.set_title("t1 vs x2 — ujemna korelacja (wyższa temp → mniejsze obciążenie)")
plt.tight_layout()
plt.show()

corr = multi.select(pl.corr("t1", "x2")).item()
print(f"Korelacja Pearsona t1 vs x2: {corr:.3f}")

## 6. Miesięczny profil x2 — kluczowy insight

Widać dramatyczny spadek od marca/kwietnia. To właśnie musimy ekstrapolować na maj–październik.

In [ ]:
monthly_profile = (
    sample.filter(pl.col("period") == "train")
    .with_columns(pl.col("timedate").dt.month().alias("month"))
    .group_by(["deviceId", "month"])
    .agg(pl.col("x2").mean().alias("x2_mean"))
    .group_by("month")
    .agg(
        [
            pl.col("x2_mean").mean().alias("x2_avg"),
            pl.col("x2_mean").quantile(0.25).alias("q25"),
            pl.col("x2_mean").quantile(0.75).alias("q75"),
        ]
    )
    .sort("month")
)

months = monthly_profile["month"].to_numpy()
avg = monthly_profile["x2_avg"].to_numpy()
q25 = monthly_profile["q25"].to_numpy()
q75 = monthly_profile["q75"].to_numpy()

month_names = {
    1: "Sty",
    2: "Lut",
    3: "Mar",
    4: "Kwi",
    5: "Maj",
    6: "Cze",
    7: "Lip",
    8: "Sie",
    9: "Wrz",
    10: "Paź",
    11: "Lis",
    12: "Gru",
}
labels = [month_names[m] for m in months]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    labels,
    avg,
    color=["steelblue" if m in [10, 11, 12, 1, 2, 3, 4] else "orange" for m in months],
    alpha=0.8,
)
ax.fill_between(range(len(months)), q25, q75, alpha=0.2, color="gray")
ax.set_ylabel("Średnie x2")
ax.set_title("Miesięczny profil x2 — dane dostępne tylko dla zaznaczonych na niebiesko")
ax.axvline(x=len(months) - 0.5, color="red", ls="--", lw=1)

# Adnotacja
for i, (m, v) in enumerate(zip(months, avg)):
    ax.text(i, v + 0.002, f"{v:.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()
print("Niebieski = dane treningowe | Pomarańczowy = cel predykcji (x2 ukryte)")

## 7. Rozkład x3 i deviceType — kluczowe dla bazowego obciążenia CWU

In [ ]:
device_info = (
    lazy.filter(pl.col("period") == "train")
    .select(["deviceId", "x3", "deviceType"])
    .group_by("deviceId")
    .agg([pl.col("x3").first(), pl.col("deviceType").first()])
    .collect()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x3_counts = device_info["x3"].value_counts().sort("x3")
axes[0].bar(
    [str(v) for v in x3_counts["x3"].to_list()],
    x3_counts["count"].to_list(),
    color="steelblue",
    alpha=0.8,
)
axes[0].set_title("x3 — typ krzywej grzewczej (proxy izolacji budynku)")
axes[0].set_xlabel("x3")
axes[0].set_ylabel("Liczba urządzeń")

dt_counts = device_info["deviceType"].value_counts().sort("deviceType")
axes[1].bar(
    [str(v) for v in dt_counts["deviceType"].to_list()],
    dt_counts["count"].to_list(),
    color="tomato",
    alpha=0.8,
)
axes[1].set_title("deviceType — kategoria urządzenia")
axes[1].set_xlabel("deviceType")

plt.tight_layout()
plt.show()

## 8. Problem ekstrapolacji — dlaczego zwykłe drzewa nie działają

LightGBM/XGBoost nie mogą przewidzieć wartości poza zakresem wartości treningowych. Jeśli minimalne miesięczne x2 w treningu to ~0.05, model nie może przewidzieć ~0.01 dla lata.

In [ ]:
# Rozkład dziennych średnich x2 per miesiąc — widać minima
month_names = {
    1: "Sty",
    2: "Lut",
    3: "Mar",
    4: "Kwi",
    5: "Maj",
    6: "Cze",
    7: "Lip",
    8: "Sie",
    9: "Wrz",
    10: "Paź",
    11: "Lis",
    12: "Gru",
}

daily_all = (
    lazy.filter(pl.col("period") == "train")
    .with_columns(
        [
            pl.col("timedate")
            .str.replace(r" UTC$", "")
            .str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
            .dt.date()
            .alias("date"),
            pl.col("timedate")
            .str.replace(r" UTC$", "")
            .str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
            .dt.month()
            .alias("month"),
        ]
    )
    .group_by(["deviceId", "date", "month"])
    .agg(pl.col("x2").mean().alias("x2_daily"))
    .collect()
)

fig, ax = plt.subplots(figsize=(12, 5))

month_order = [10, 11, 12, 1, 2, 3, 4]
colors = plt.cm.coolwarm(np.linspace(0, 1, len(month_order)))

for i, m in enumerate(month_order):
    vals = daily_all.filter(pl.col("month") == m)["x2_daily"].drop_nulls().to_numpy()
    if len(vals) == 0:
        continue
    ax.boxplot(
        vals,
        positions=[i],
        widths=0.6,
        patch_artist=True,
        boxprops=dict(facecolor=colors[i], alpha=0.7),
        medianprops=dict(color="black", lw=2),
        flierprops=dict(marker=".", markersize=1, alpha=0.3),
    )

ax.set_xticks(range(len(month_order)))
ax.set_xticklabels([f"{month_names[m]}\n({m})" for m in month_order])
ax.set_ylabel("x2 (dzienna średnia)")
ax.set_title("Rozkład dziennego x2 per miesiąc (dane treningowe)")
q01 = float(daily_all["x2_daily"].drop_nulls().quantile(0.01) or 0)
ax.axhline(y=q01, color="red", ls="--", lw=1, label=f"1. percentyl = {q01:.4f} ← minimum dla drzew")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Strategia solution.py — pipeline w skrócie

```
data.csv (10GB, 5-min)  ──►  czyszczenie (stuck meters, outliers)
devices.csv             ──►  join lat/lon
Open-Meteo API          ──►  absolutna temperatura [°C] → HDD, COP, load_proxy
                                                         ↓
                        agregacja do dziennej granularności
                                                         ↓
                 ┌──────────────────────────────────────┐
                 │  3-fold temporal CV (Oct→Feb, Mar, Apr)│
                 │  Model A: LightGBM (huber, linear_tree)│
                 │  Model B: LightGBM (log1p x2)          │
                 │  Model C: Ridge (physics features)      │
                 │  Model D: per-device OLS (HDD→x2)       │
                 └──────────────────────────────────────┘
                                                         ↓
                 mediana ensemble → agregacja do miesięcy
                                                         ↓
                 post-processing: DHW floor, Oct anchor, clip
                                                         ↓
                              submission.csv
```

**Kluczowa cecha**: `load_proxy = HDD_15 × inverse_COP`  
Latem HDD=0 → load_proxy=0 → model naturalnie przewiduje tylko bazowe CWU.

In [ ]:
# Symulacja load_proxy dla różnych temperatur
T_outdoor = np.linspace(-15, 30, 200)
T_base = 15.0
T_indoor = 20.0

HDD = np.maximum(T_base - T_outdoor, 0)
inv_cop = np.maximum((T_indoor - T_outdoor) / T_indoor, 0)
load_proxy = HDD * inv_cop

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(T_outdoor, HDD, color="steelblue", lw=2)
axes[0].axvline(x=15, color="red", ls="--", lw=1)
axes[0].set_title("HDD_15 (Heating Degree Days)")
axes[0].set_xlabel("Temperatura zewnętrzna [°C]")
axes[0].set_ylabel("HDD")

axes[1].plot(T_outdoor, inv_cop, color="tomato", lw=2)
axes[1].axvline(x=20, color="red", ls="--", lw=1)
axes[1].set_title("inverse_COP_proxy")
axes[1].set_xlabel("Temperatura zewnętrzna [°C]")

axes[2].plot(T_outdoor, load_proxy, color="purple", lw=2)
axes[2].axvline(x=15, color="red", ls="--", lw=1, label="T=15°C → load_proxy=0")
axes[2].set_title("load_proxy = HDD × inv_COP\n(=0 latem!)")
axes[2].set_xlabel("Temperatura zewnętrzna [°C]")
axes[2].legend(fontsize=8)

plt.suptitle("Ekstrapolacja przez fizykę — load_proxy naturalnie zanika latem", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 10. Walidacja CV — symulacja problemu sezonowego

Nie możemy używać k-fold CV — musimy symulować przejście z zimy na lato.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))

# Oś czasu: Okt 2024 → Okt 2025
all_months = [
    "Paź\n2024",
    "Lis",
    "Gru",
    "Sty\n2025",
    "Lut",
    "Mar",
    "Kwi",
    "Maj",
    "Cze",
    "Lip",
    "Sie",
    "Wrz",
    "Paź\n2025",
]
n = len(all_months)

# Kolory: train=niebieski, val=zielony, test=pomarańczowy, predykcja=czerwony
colors_map = {"train": "#4C72B0", "val": "#55A868", "fold_val": "#C44E52", "predict": "#DD8452"}

# Dane treningowe (okt-kwi)
for i in range(7):
    ax.barh(0, 1, left=i, height=0.3, color=colors_map["train"], alpha=0.6)

# Fold 1: walidacja luty
ax.barh(-0.5, 4, left=0, height=0.25, color=colors_map["train"], alpha=0.4)
ax.barh(-0.5, 1, left=4, height=0.25, color=colors_map["fold_val"], alpha=0.9)
ax.text(4.5, -0.5, "F1", ha="center", va="center", color="white", fontsize=8, fontweight="bold")

# Fold 2: walidacja marzec
ax.barh(-0.85, 5, left=0, height=0.25, color=colors_map["train"], alpha=0.4)
ax.barh(-0.85, 1, left=5, height=0.25, color=colors_map["fold_val"], alpha=0.9)
ax.text(5.5, -0.85, "F2", ha="center", va="center", color="white", fontsize=8, fontweight="bold")

# Fold 3: walidacja kwiecień (KLUCZOWE)
ax.barh(-1.2, 6, left=0, height=0.25, color=colors_map["train"], alpha=0.4)
ax.barh(-1.2, 1, left=6, height=0.25, color="#8B0000", alpha=0.9)
ax.text(6.5, -1.2, "F3★", ha="center", va="center", color="white", fontsize=8, fontweight="bold")

# Predykcja: maj-październik
for i in range(7, 13):
    ax.barh(0, 1, left=i, height=0.3, color=colors_map["predict"], alpha=0.7)

ax.set_xlim(0, n)
ax.set_xticks(range(n))
ax.set_xticklabels(all_months, fontsize=9)
ax.set_yticks([0, -0.5, -0.85, -1.2])
ax.set_yticklabels(["Dane", "Fold 1", "Fold 2", "Fold 3 ★"], fontsize=9)
ax.set_title("Temporal Forward-Chaining CV — symuluje przejście zimа→lato")

from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=colors_map["train"], label="Train"),
    Patch(facecolor=colors_map["fold_val"], label="Fold validation"),
    Patch(facecolor="#8B0000", label="Fold 3 (kwiecień = najważniejszy)"),
    Patch(facecolor=colors_map["predict"], label="Predykcja (x2 ukryte)"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## 11. Podsumowanie — czego szukamy w danych

| Cecha | Skąd | Dlaczego ważna |
|---|---|---|
| `load_proxy = HDD_15 × inv_COP` | Open-Meteo | Naturalnie → 0 latem |
| `temp_mean` | Open-Meteo | Główny driver, monotoniczny |
| `day_length` | astronomia | Deterministyczny sygnał sezonowy |
| `HDD_15`, `HDD_18` | Open-Meteo | Stopniodni grzania |
| `CDD_22`, `CDD_24` | Open-Meteo | Dla ewentualnego chłodzenia |
| `x3` | data.csv | Proxy izolacji budynku → bazowe CWU |
| `deviceType` | data.csv | Kategoryzacja (pompa powietrze/grunt/CWU) |
| `t7_mean` | data.csv | Temperatura zbiornika CWU |

**Cel MAE**: niższe = lepsze. Przewaga naszego podejścia: `load_proxy` pozwala modelowi przewidywać właściwe wartości letnie bez zobaczenia ich w treningu.